# Chapter 8 &mdash; The Postage Stamp Problem and Sylvester's Formula

**Concept 9 of the Chapter 8 decomposition:** *The Postage Stamp Problem, the Frobenius Number, and Sylvester's Formula*

$Fr(p,q)=pq-p-q$ for relatively prime $p,q$ &mdash; the largest amount you cannot pay.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Postage-Stamp-Frobenius/Concept-Postage-Stamp-Frobenius.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


With only $p$-cent and $q$-cent stamps, which amounts can you make? If $p$ and $q$ are
**relatively prime**, all sufficiently large amounts are payable, and the largest
**un**payable one is the **Frobenius number**

$$Fr(p,q) = pq - p - q \qquad \text{(Sylvester)}.$$

For $p=3, q=5$ that is $15-3-5=7$: you cannot make 7 cents, but you can make 8, 9, 10
and everything above.

If $p$ and $q$ share a factor, infinitely many amounts are unpayable and no Frobenius
number exists. That relative-primality condition is doing real work.

## 2. Definitions

### Payable amounts, by brute force

In [ ]:
def payable(p, q, upto):
    return sorted({p*i + q*j for i in range(upto // p + 1)
                             for j in range(upto // q + 1) if p*i + q*j <= upto})

def frobenius_bruteforce(p, q, upto=400):
    pay = set(payable(p, q, upto))
    gaps = [n for n in range(upto) if n not in pay]
    return max(gaps) if gaps else None

### Sylvester's formula

In [ ]:
from math import gcd
def sylvester(p, q):
    return p*q - p - q if gcd(p, q) == 1 else None

<!-- nav-strip -->

---

&larr;&nbsp;[Ch8&nbsp;8.&nbsp;Cross-Checking Two Designs by Minimal-DFA Isomorphism](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Cross-Check-By-Isomorphism/Concept-Cross-Check-By-Isomorphism.ipynb) &nbsp;&middot;&nbsp; [**Chapter 8** index](https://github.com/ganeshutah/Jove/blob/master/Chapter8-RE/README.md) &nbsp;&middot;&nbsp; [Ch8&nbsp;10.&nbsp;Ultimately Periodic Sets, and the Lengths of Strings in a Regular Language](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Ultimately-Periodic-Sets/Concept-Ultimately-Periodic-Sets.ipynb)&nbsp;&rarr;

---

## 3. Tests

For $p=3,q=5$: everything from 8 up is payable, and 7 is not.

In [ ]:
pay = set(payable(3, 5, 30))
print("payable up to 30 :", sorted(pay))
print("gaps             :", [n for n in range(30) if n not in pay])
assert 7 not in pay and all(n in pay for n in range(8, 31))

Sylvester's formula matches brute force, for every coprime pair tried.

In [ ]:
from math import gcd
for p, q in [(3,5), (5,7), (4,7), (3,7), (5,9), (7,11)]:
    if gcd(p, q) != 1: continue
    bf, sy = frobenius_bruteforce(p, q), sylvester(p, q)
    print("Fr(%2d,%2d) : brute force %3d, Sylvester %3d" % (p, q, bf, sy))
    assert bf == sy

Without relative primality there is **no** Frobenius number.

In [ ]:
p, q = 4, 6
print("gcd(%d,%d) = %d" % (p, q, gcd(p, q)))
pay = set(payable(p, q, 60))
odd_gaps = [n for n in range(60) if n not in pay]
print("unpayable amounts below 60 :", odd_gaps[:14], "...", "count", len(odd_gaps))
print("Sylvester returns :", sylvester(p, q))
assert sylvester(p, q) is None
print("\nEvery odd amount is unpayable forever -- the set of gaps is infinite.")

The payable set is **ultimately periodic**: beyond $Fr$, every amount works.

In [ ]:
p, q = 5, 7
fr = sylvester(p, q)
pay = set(payable(p, q, 200))
assert all(n in pay for n in range(fr + 1, 200))
print("Fr(5,7) = %d ; every amount from %d to 199 is payable" % (fr, fr + 1))
print("That 'everything beyond a bound' shape is Concept 10.")

## 4. Exercises


1. Compute $Fr(11,13)$ two ways.
2. Why does relative primality guarantee *some* bound exists?
3. Is there a closed formula for three stamp denominations? Look it up.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter8-RE/Concept-Postage-Stamp-Frobenius')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')